<a href="https://colab.research.google.com/github/tr33yut/-Netflix-/blob/main/Project_5001_Clean_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

input_path = r"filtered_box_office_movies 2.csv"
df = pd.read_csv(input_path)

print(f"จำนวนแถวก่อนรวมข้อมูล: {len(df)} แถว")

numeric_cols = ['weekly_hours_viewed', 'weekly_views', 'weekly_rank', 'cumulative_weeks_in_top_10']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')

# 3. กำหนดเงื่อนไขการรวมข้อมูล (Aggregation Rules)
agg_rules = {
    'category': 'first',
    'weekly_rank': 'min',                         # อันดับที่ดีที่สุด (เช่น เคยได้ที่ 1)
    'weekly_hours_viewed': 'sum',                 # รวมชั่วโมงรับชมทั้งหมดทุกสัปดาห์
    'weekly_views': 'sum',                        # รวมยอดวิวทั้งหมดทุกสัปดาห์
    'cumulative_weeks_in_top_10': 'max',          # จำนวนสัปดาห์สูงสุดที่ติด Top 10
    'runtime': 'first',
    'Budget_USD': 'first',
    'Box_Office_USD': 'first'
}

agg_rules = {k: v for k, v in agg_rules.items() if k in df.columns}

df_unique = df.groupby('show_title', as_index=False).agg(agg_rules)

df_unique.rename(columns={
    'weekly_rank': 'best_rank',
    'weekly_hours_viewed': 'total_hours_viewed',
    'weekly_views': 'total_views',
    'cumulative_weeks_in_top_10': 'total_weeks_in_top10'
}, inplace=True)

output_path = r"movies_aggregated_unique 3.csv"
df_unique.to_csv(output_path, index=False)

print(f"รวมข้อมูลเสร็จสิ้น เหลือหนังที่ไม่ซ้ำกัน: {len(df_unique)} เรื่อง")
print(f"บันทึกไฟล์ไว้ที่: {output_path}")
df_unique.head(5)

จำนวนแถวก่อนรวมข้อมูล: 2130 แถว
รวมข้อมูลเสร็จสิ้น เหลือหนังที่ไม่ซ้ำกัน: 883 เรื่อง
บันทึกไฟล์ไว้ที่: movies_aggregated_unique 3.csv


,show_title,category,best_rank,total_hours_viewed,total_views,total_weeks_in_top10,runtime,Budget_USD,Box_Office_USD
0,'83,Films (Non-English),4,13650000,8.531250e+04,2,160.0000,36786988,25452983
1,100 Meters,Films (Non-English),6,4600000,2.600000e+06,2,1.7833,No Data / Netflix Original,3611916
2,12 Strong,Films (English),6,16420000,1.263077e+05,2,130.0000,35000000,67450815
3,13 Hours: The Secret Soldiers of Benghazi,Films (English),3,47950000,9.182292e+06,4,2.4000,50000000,69411370
4,1917,Films (English),7,10840000,9.109244e+04,1,119.0000,100000000,446064352


In [ ]:
import pandas as pd
import requests

input_path = r"movies_with_imdb_id_final 4.csv"
output_path = r"movies_with_ratings_complete 5.csv"

print("กำลังอ่านไฟล์ตั้งต้น...")
df = pd.read_csv(input_path, encoding='utf-8-sig')

ratings_url = "https://datasets.imdbws.com/title.ratings.tsv.gz"
local_gz_path = r"D:\5001\Project\title.ratings.tsv.gz"

print("กำลังดาวน์โหลดชุดข้อมูลคะแนนอย่างเป็นทางการจาก IMDb...")
res = requests.get(ratings_url, stream=True, timeout=30)
with open(local_gz_path, "wb") as f:
    for chunk in res.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("ดาวน์โหลดสำเร็จ! กำลังรวมคะแนนเข้ากับรายชื่อหนัง...")

ratings_df = pd.read_csv(local_gz_path, sep='\t', compression='gzip')

merged_df = pd.merge(
    df,
    ratings_df[['tconst', 'averageRating', 'numVotes']],
    left_on='imdb_id',
    right_on='tconst',
    how='left'
)

merged_df = merged_df.rename(columns={
    'averageRating': 'imdb_rating',
    'numVotes': 'imdb_total_votes'
}).drop(columns=['tconst'])

merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')

success_count = merged_df['imdb_rating'].notna().sum()
print(f"\n รวมข้อมูลคะแนนสำเร็จเรียบร้อย!")
print(f"- พบข้อมูลคะแนน: {success_count} จากทั้งหมด {len(merged_df)} เรื่อง")
print(f"- บันทึกไฟล์ไว้ที่: {output_path}")

merged_df[['show_title', 'Box_Office_USD', 'imdb_rating', 'imdb_total_votes']].head(5)

กำลังอ่านไฟล์ตั้งต้น...
กำลังดาวน์โหลดชุดข้อมูลคะแนนอย่างเป็นทางการจาก IMDb...
ดาวน์โหลดสำเร็จ! กำลังรวมคะแนนเข้ากับรายชื่อหนัง...

 รวมข้อมูลคะแนนสำเร็จเรียบร้อย!
- พบข้อมูลคะแนน: 868 จากทั้งหมด 881 เรื่อง
- บันทึกไฟล์ไว้ที่: movies_with_ratings_complete 5.csv


,show_title,Box_Office_USD,imdb_rating,imdb_total_votes
0,'83,25452983,7.5,43430.0
1,100 Meters,3611916,7.6,8869.0
2,12 Strong,67450815,6.6,102075.0
3,13 Hours: The Secret Soldiers of Benghazi,69411370,7.3,182498.0
4,1917,446064352,8.2,786332.0


In [ ]:
import pandas as pd
import requests
import re
import time

# 1. โหลดไฟล์ข้อมูลที่คลีนแล้ว (881 เรื่อง)
input_path = r"movies_aggregated_unique 3.csv"
output_path = r"movies_with_imdb_id_final 4.csv"

print(" กำลังโหลดไฟล์ข้อมูล...")
df = pd.read_csv(input_path, encoding='utf-8')
print(f"โหลดข้อมูลสำเร็จ ทั้งหมด {len(df)} เรื่อง\n")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def fetch_imdb_id(title):
    """ค้นหา IMDb ID โดยรองรับทั้ง slug และ direct query"""
    clean_title = re.sub(r"^[^a-zA-Z0-9]+", "", str(title))
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", clean_title).lower().strip('_')
    first_char = slug[0] if slug else 'a'

    # ยิง 2 รูปแบบ URL เพื่อให้ครอบคลุมชื่อหนังทุกประเภท
    target_urls = [
        f"https://v2.sg.media-imdb.com/suggestion/{first_char}/{slug}.json",
        f"https://v2.sg.media-imdb.com/suggestion/t/{requests.utils.quote(str(title))}.json"
    ]

    for url in target_urls:
        try:
            res = requests.get(url, headers=headers, timeout=6)
            if res.status_code == 200:
                data = res.json()
                for item in data.get('d', []):
                    # กรองเฉพาะรหัสที่เป็นภาพยนตร์/ซีรีส์ (ขึ้นต้นด้วย tt)
                    if item.get('id', '').startswith('tt'):
                        return item['id']
        except Exception:
            continue
    return None

# 2. เริ่มลูปดึงรหัส IMDb ID
imdb_ids = []
total = len(df)
print(f"🚀 เริ่มต้นการดึง IMDb ID ทั้งหมด {total} เรื่อง (ใช้เวลาประมาณ 5-6 นาที)...\n")

for i, title in enumerate(df['show_title'], start=1):
    movie_id = fetch_imdb_id(title)
    imdb_ids.append(movie_id)

    status = f" พบ ID: {movie_id}" if movie_id else "❌ ไม่พบ ID"
    print(f"[{i}/{total}] {title} -> {status}")

    # หน่วงเวลา 0.35 วินาที เพื่อความปลอดภัยไม่ให้โดนบล็อก
    time.sleep(0.35)

    # เซฟสำรองข้อมูลทุกๆ 50 เรื่อง ป้องกันกรณีเน็ตหลุดระหว่างทาง
    if i % 50 == 0:
        df_temp = df.iloc[:i].copy()
        df_temp['imdb_id'] = imdb_ids
        df_temp.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"--- บันทึกข้อมูลสำรองชั่วคราวแล้ว ({i}/{total} เรื่อง) ---\n")

# 3. บันทึกผลลัพธ์ไฟล์สมบูรณ์
df['imdb_id'] = imdb_ids
df.to_csv(output_path, index=False, encoding='utf-8-sig')

found_count = df['imdb_id'].notna().sum()
print(f"\n ดึงข้อมูลเสร็จสิ้น!")
print(f"- สำเร็จ: {found_count}/{total} เรื่อง")
print(f"- บันทึกไฟล์ไว้ที่: {output_path}")
df[['show_title', 'best_rank', 'Box_Office_USD', 'imdb_id']].head(10)

 กำลังโหลดไฟล์ข้อมูล...
โหลดข้อมูลสำเร็จ ทั้งหมด 883 เรื่อง

🚀 เริ่มต้นการดึง IMDb ID ทั้งหมด 883 เรื่อง (ใช้เวลาประมาณ 5-6 นาที)...

[1/883] '83 ->  พบ ID: tt7518786
[2/883] 100 Meters ->  พบ ID: tt32600395
[3/883] 12 Strong ->  พบ ID: tt1413492
[4/883] 13 Hours: The Secret Soldiers of Benghazi ->  พบ ID: tt4172430
[5/883] 1917 ->  พบ ID: tt8579674
[6/883] 2 Guns ->  พบ ID: tt1272878
[7/883] 21 Jump Street ->  พบ ID: tt1232829
[8/883] 211 ->  พบ ID: tt4976192
[9/883] 27 Dresses ->  พบ ID: tt0988595
[10/883] 28 Weeks Later ->  พบ ID: tt0463854
[11/883] 28 Years Later ->  พบ ID: tt10548174
[12/883] 365 Days ->  พบ ID: tt10886166
[13/883] 4 Kings 2 ->  พบ ID: tt30195433
[14/883] 40 Acres ->  พบ ID: tt29634843
[15/883] 47 Meters Down: Uncaged ->  พบ ID: tt7329656
[16/883] 47 Ronin ->  พบ ID: tt1335975
[17/883] 65 ->  พบ ID: tt12261776
[18/883] A Bad Moms Christmas ->  พบ ID: tt6359956
[19/883] A Boy Called Christmas ->  พบ ID: tt10187208
[20/883] A Brother and 7 Siblings ->  พบ ID: tt3288

,show_title,best_rank,Box_Office_USD,imdb_id
0,'83,4,25452983,tt7518786
1,100 Meters,6,3611916,tt32600395
2,12 Strong,6,67450815,tt1413492
3,13 Hours: The Secret Soldiers of Benghazi,3,69411370,tt4172430
4,1917,7,446064352,tt8579674
5,2 Guns,2,131940411,tt1272878
6,21 Jump Street,6,201585328,tt1232829
7,211,6,1052222,tt4976192
8,27 Dresses,7,160300000,tt0988595
9,28 Weeks Later,6,72304846,tt0463854
